# Banking Incident Classification: v1

**Question:** Can we tell which hours were critical incidents from the system's KPIs, and can the previous hour's KPIs warn us early?

**Data:** `incident_reports.csv` (168 hourly rows, 2025-10-20 to 2025-10-26) is the labeled file. `infrastructure_metrics.csv` (10,080 minute rows, same week) is used only for data checks and the mismatch comparison.

**Plan:** (1) check the data, (2) see how the labels relate to the KPIs, (3) run a time-based baseline and model, (4) test whether the previous hour predicts the next hour's label.

## 1. Setup and load

In [ ]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

inc = pd.read_csv('incident_reports.csv', parse_dates=['timestamp']).sort_values('timestamp').reset_index(drop=True)
infra = pd.read_csv('infrastructure_metrics.csv', parse_dates=['timestamp'])

KPIS = ['cpu_utilization', 'memory_usage', 'db_connection_pool_usage', 'application_error_rate', 'neft_success_rate']
TARGET = 'is_critical'
print(inc.shape, infra.shape)
inc.head()

(168, 14) (10080, 13)


,incident_id,timestamp,cpu_utilization,memory_usage,db_connection_pool_usage,network_throughput_mbps,application_error_rate,neft_success_rate,imps_success_rate,rtgs_success_rate,avg_transaction_processing_time_ms,incident_type,severity,is_critical
0,INC000001,2025-10-20 00:00:00,42.82,36.96,25.62,168.08,0,99.46,99.20,99.71,108.99,Normal,Low,0
1,INC000002,2025-10-20 01:00:00,38.23,39.58,33.64,215.29,2,96.62,98.36,99.86,100.11,Normal,Low,0
2,INC000003,2025-10-20 02:00:00,35.37,38.81,22.94,218.44,2,96.49,98.61,99.10,100.00,Normal,Low,0
3,INC000004,2025-10-20 03:00:00,36.81,37.07,39.59,185.52,3,98.56,99.96,99.60,100.00,Normal,Low,0
4,INC000005,2025-10-20 04:00:00,43.25,33.70,39.36,215.06,4,99.96,99.51,99.44,140.75,Normal,Low,0


## 2. Data checks
Missing values, duplicates, and gaps in both files.

In [ ]:
print('Incident file missing values:', inc.isna().sum().sum())
print('Infra file missing values:', infra.isna().sum().sum())
print('Infra duplicate timestamps:', infra.timestamp.duplicated().sum())
print('Infra gaps (not 1 min):', (infra.timestamp.diff().dropna() != pd.Timedelta('1min')).sum())
print('Incident gaps (not 1 h):', (inc.timestamp.diff().dropna() != pd.Timedelta('1h')).sum())
print('Label counts:', inc[TARGET].value_counts().to_dict())

Incident file missing values: 0
Infra file missing values: 0
Infra duplicate timestamps: 0
Infra gaps (not 1 min): 0
Incident gaps (not 1 h): 0
Label counts: {0: 142, 1: 26}


## 3. Normal vs. critical hours
If critical hours look very different, the labels are probably tied to these KPIs.

In [ ]:
inc.groupby(TARGET)[KPIS].agg(['min', 'max']).round(2).T

is_critical                       0      1
cpu_utilization          min  34.03  85.41
                         max  76.38  94.81
memory_usage             min  32.21  80.84
                         max  66.26  89.91
db_connection_pool_usage min  20.13  85.13
                         max  66.58  93.75
application_error_rate   min   0.00  10.00
                         max   6.00  19.00
neft_success_rate        min  96.00  85.11
                         max  99.98  89.76

**Reading the table:** Every KPI's critical range sits entirely on one side of its normal range. CPU, memory, connection pool, and error rate are all higher in critical hours. NEFT success is lower.

Next, check how many KPIs separate the classes on their own.

In [ ]:
rows = []
for f in KPIS:
    n, c = inc.loc[inc[TARGET]==0, f], inc.loc[inc[TARGET]==1, f]
    if c.min() > n.max():
        rows.append((f, f'> {n.max():.2f}', 'perfect'))
    elif c.max() < n.min():
        rows.append((f, f'< {n.min():.2f}', 'perfect'))
    else:
        rows.append((f, '-', 'overlaps'))
pd.DataFrame(rows, columns=['KPI', 'Critical if', 'Separation'])

,KPI,Critical if,Separation
0,cpu_utilization,> 76.38,perfect
1,memory_usage,> 66.26,perfect
2,db_connection_pool_usage,> 66.58,perfect
3,application_error_rate,> 6.00,perfect
4,neft_success_rate,< 96.00,perfect


**Finding 1:** Each KPI separates the classes perfectly by itself. This means the labels are likely rule-based, and a learned model on the same hour can't show much skill.

## 4. File mismatch
Do the incident and minute-level files agree at the same hours?

In [ ]:
m = inc.set_index('timestamp').join(infra.set_index('timestamp')[['neft_success_rate', 'cpu_utilization']], rsuffix='_infra')
m['match'] = (m.neft_success_rate - m.neft_success_rate_infra).abs() < 0.01
print(pd.crosstab(m.incident_type, m.match))
crit = m[m[TARGET]==1]
print('\nAt critical hours (mean):')
print(crit[['neft_success_rate', 'neft_success_rate_infra', 'cpu_utilization', 'cpu_utilization_infra']].mean().round(2).to_string())

match                False  True 
incident_type                    
Normal                   0    139
Resource_Exhaustion     26      0
Resource_Warning         0      3

At critical hours (mean):
neft_success_rate          87.32
neft_success_rate_infra    98.12
cpu_utilization            89.84
cpu_utilization_infra      49.38


**Finding 2:** The two files agree at 142 hours and disagree at all 26 critical hours. The incident file shows NEFT near 87% and CPU near 90%. The minute-level file shows NEFT near 98% at those hours. v1 uses only the incident file. The conflict needs to be resolved with the instructor.

## 5. Time-based split
Train on the first 5 days, test on the last 2. No shuffling, since these are time series.

In [ ]:
cutoff = inc.timestamp.min() + pd.Timedelta(days=5)
train, test = inc[inc.timestamp < cutoff], inc[inc.timestamp >= cutoff]
print(f'Train: {len(train)} hours, {train[TARGET].sum()} critical')
print(f'Test:  {len(test)} hours, {test[TARGET].sum()} critical')

Train: 120 hours, 17 critical
Test:  48 hours, 9 critical


## 6. Same-hour detection
Predict each hour's label from that hour's KPIs. This is the task as originally defined.

In [ ]:
def report(name, y_true, y_pred):
    print(name)
    print(f'  precision={precision_score(y_true, y_pred, zero_division=0):.3f}  '
          f'recall={recall_score(y_true, y_pred, zero_division=0):.3f}  '
          f'F1={f1_score(y_true, y_pred, zero_division=0):.3f}')
    print(confusion_matrix(y_true, y_pred), '\n')

y_test = test[TARGET]
report('Threshold rule: critical if NEFT < 95%', y_test, (test.neft_success_rate < 95).astype(int))

no_neft = [k for k in KPIS if k != 'neft_success_rate']
for name, feats in [('Logistic regression, all KPIs', KPIS), ('Logistic regression, without NEFT', no_neft)]:
    model = make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=1000))
    model.fit(train[feats], train[TARGET])
    report(name, y_test, model.predict(test[feats]))

Threshold rule: critical if NEFT < 95%
  precision=1.000  recall=1.000  F1=1.000
[[39  0]
 [ 0  9]] 

Logistic regression, all KPIs
  precision=1.000  recall=1.000  F1=1.000
[[39  0]
 [ 0  9]] 

Logistic regression, without NEFT
  precision=1.000  recall=1.000  F1=1.000
[[39  0]
 [ 0  9]] 



**Finding 3:** Every same-hour method gets all 48 test hours right. This is expected given Finding 1. It shows the labels can be recovered from the KPIs, not that a model has learned an early-warning pattern.

## 7. Early warning: previous hour predicts this hour
Use the previous hour's KPIs to predict whether this hour is critical. This is the real forecasting test.

In [ ]:
lagged = inc[KPIS].shift(1).add_suffix('_prev')
d = pd.concat([inc[['timestamp', TARGET]], lagged], axis=1).dropna().reset_index(drop=True)
tr = d[d.timestamp < cutoff]
te = d[d.timestamp >= cutoff]
LAG_FEATS = [c for c in d.columns if c.endswith('_prev')]
print(f'Train: {len(tr)} rows, {tr[TARGET].sum()} critical | Test: {len(te)} rows, {te[TARGET].sum()} critical')

report('Persistence: critical if previous hour NEFT < 95%', te[TARGET], (te.neft_success_rate_prev < 95).astype(int))

lag_no_neft = [c for c in LAG_FEATS if 'neft' not in c]
for name, feats in [('Logistic regression, lagged, all KPIs', LAG_FEATS), ('Logistic regression, lagged, without NEFT', lag_no_neft)]:
    model = make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=1000))
    model.fit(tr[feats], tr[TARGET])
    report(name, te[TARGET], model.predict(te[feats]))

Train: 119 rows, 17 critical | Test: 48 rows, 9 critical
Persistence: critical if previous hour NEFT < 95%
  precision=0.000  recall=0.000  F1=0.000
[[30  9]
 [ 9  0]] 

Logistic regression, lagged, all KPIs
  precision=0.214  recall=0.333  F1=0.261
[[28 11]
 [ 6  3]] 

Logistic regression, lagged, without NEFT
  precision=0.176  recall=0.333  F1=0.231
[[25 14]
 [ 6  3]] 



**Finding 4:** Early warning is weak. The lagged model catches 3 of the 9 critical test hours, with 11 false alarms. Persistence catches none. Critical hours start abruptly, so the previous hour usually looks normal.

## 8. Summary
- Current-hour labels are recoverable by a single KPI threshold, so classifier scores here show the labeling rule, not learned skill.
- Predicting from the previous hour is hard. The best lagged model catches 3 of 9 critical test hours (recall 0.33, F1 0.26) with 11 false alarms.
- The two source files disagree at every critical hour. v1 is built on the incident file.
- Open question for the instructor: how the labels were created and which file is authoritative.